# A LoRA that walks a library instead of memorising it

**lora-kernel · `distributor-wiki@v2` on `gemma-4-E4B-it`** — [repository](https://github.com/EvolvingAgentsLabs/lora-kernel) · [adapter](https://huggingface.co/Matias/lora-kernel-distributor-wiki-gemma4-e4b)

The adapter holds no facts. It learned a habit: `<search>` the wiki, `<open>` a page, open the one section it needs, follow the link inside the sentence, and cite the statement its answer rests on. The runtime writes each result right after the tag, and generation continues.

This notebook runs three things:
1. **The walk.** The same multi-hop questions, answered by the bare model and then by the model with the adapter, on a wiki neither was trained on.
2. **The grade.** A reply earns credit only if the value is right *and* the cited section actually says it.
3. **Editing a note.** We change a number in one markdown page and ask again. The answer follows the page and nothing is retrained.

**Runtime:** GPU with bf16 (L4 or A100). About 10 minutes, most of it the model download.

In [ ]:
# 1 · Install and clone. The runtime (memory/) and the grader live in the repository.
# Colab ships an old torchao that current PEFT refuses to load next to; this notebook does not use it.
!pip -q uninstall -y torchao
!pip -q install -U transformers peft accelerate
!git clone -q --depth 1 https://github.com/EvolvingAgentsLabs/lora-kernel.git
%cd lora-kernel

In [ ]:
# 2 · Load the base in bf16, the way the numbers on the model card were served, and the adapter on it.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "google/gemma-4-E4B-it"
ADAPTER = "Matias/lora-kernel-distributor-wiki-gemma4-e4b"
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), \
    "this notebook needs a bf16 GPU (L4 or A100): the adapter was measured in bf16, and a T4 has no bf16"

tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16, device_map="cuda")
model = PeftModel.from_pretrained(base, ADAPTER).eval()

# The adapter must be applied, or every number below measures the bare model: 516 tensors = 258 LoRA modules.
lora = [n for n, m in model.named_modules() if hasattr(m, "lora_A") and "default" in getattr(m, "lora_A", {})]
print(f"LoRA modules applied: {len(lora)}")
assert len(lora) == 258, "the adapter did not attach to the text stack"


In [ ]:
# 3 · The model as a function of the text so far: generate until the next closing tag, then hand back.
import os, sys
sys.path.insert(0, os.getcwd())         # the cloned repository: memory/, training/
from memory import prompt
from memory.notes import Library
from memory.runtime import ChainSuite, Conversation
from training.harness.accept_rank import run_chain
from training.wiki import grade as gr
import json, zlib, contextlib

STOPS = list(ChainSuite.close)            # </search>, </open>, </calc>

def gen_for(question, adapter=True):
    head = tok.apply_chat_template(
        [{"role": "system", "content": prompt.SYSTEM_WIKI},
         {"role": "user", "content": prompt.user_text_wiki(question)}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)   # thinking off, as trained
    def gen(prefix):
        ids = tok(head + prefix, return_tensors="pt", add_special_tokens=False).to(model.device)
        ctx = contextlib.nullcontext() if adapter else model.disable_adapter()
        with ctx, torch.no_grad():
            out = model.generate(**ids, max_new_tokens=256, do_sample=False,
                                 stop_strings=STOPS, tokenizer=tok)
        return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen

def walk(lib, row, gen):
    conv = Conversation(lib, seed=zlib.crc32(row["case_id"].encode()), mode="strict", max_opens=24)
    suite = ChainSuite(conv)
    chain = run_chain(suite.wrap(gen), {}, max_calls=24, suite=suite)
    final = chain["spans"][-1]["text"] if chain["spans"] else ""
    return gr.grade(row, final, conv), chain["text"]


In [ ]:
# 4 · The walk: a few multi-hop questions from the evaluation world, bare model against adapter.
lib = Library.load("knowledge/distributor-wiki")
rows = [json.loads(l) for l in open("training/wiki/data/eval.jsonl")]
picked = [r for r in rows if r["hops"] >= 2 and r["check"]["kind"] == "value"][:N_QUESTIONS] \
    if "N_QUESTIONS" in globals() else [r for r in rows if r["hops"] >= 2 and r["check"]["kind"] == "value"][:6]

score = {"bare": 0, "adapter": 0}
for r in picked:
    print(f"\n### {r['question']}   (expected: {r['answer']}, {r['hops']} hops)")
    for arm, on in (("bare", False), ("adapter", True)):
        g, text = walk(lib, r, gen_for(r["question"], adapter=on))
        score[arm] += g["state"] == "right"
        print(f"--- {arm}: {g['state']}\n{text.strip()[-700:]}")
print(f"\ncredit (value right AND citation verified): bare {score['bare']}/{len(picked)} · adapter {score['adapter']}/{len(picked)}")


## Editing a note, without retraining

The fact lives in a markdown file. We change the number in the statement the answer rests on, reload the library and ask again. A model that had memorised the fact would give the old number. This one reads the page.

In [ ]:
# 5 · Change one statement's number in a copy of the library and ask again.
import shutil, re, pathlib
row = next(r for r in picked if r["check"]["tokens"] and r["check"]["tokens"][0].isdigit())
nid, anchor = row["support"]
old = row["check"]["tokens"][0]
new = str(int(old) + 7)

shutil.rmtree("/tmp/edited-wiki", ignore_errors=True)
shutil.copytree("knowledge/distributor-wiki", "/tmp/edited-wiki")
page = pathlib.Path("/tmp/edited-wiki") / lib[nid].path.relative_to(lib.root)
lines = page.read_text().splitlines()
i = next(k for k, l in enumerate(lines) if l.startswith(f"§{anchor} "))
print("before:", lines[i])
lines[i] = re.sub(rf"\b{old}\b", new, lines[i], count=1)
print("after: ", lines[i])
page.write_text("\n".join(lines) + "\n")

edited = Library.load("/tmp/edited-wiki")
row_new = {**row, "answer": row["answer"].replace(old, new), "check": {**row["check"], "tokens": [new]}}
g, text = walk(edited, row_new, gen_for(row["question"], adapter=True))
print(f"\n{row['question']}\n{text.strip()[-500:]}\n\nexpected now: {new} -> {g['state']}")


## What this shows, and what it doesn't

- **Shown:** on a wiki the adapter never saw in training (its corpus came from 32 other generated worlds), it walks multi-hop questions and cites the statement it used. The full evaluation is 39/40 against the bare model's 19/40 ([`releases/distributor-wiki@v2.json`](https://github.com/EvolvingAgentsLabs/lora-kernel/blob/main/releases/distributor-wiki%40v2.json)). The six questions here are a sample, not that measurement. The full score was measured in vLLM, and plain transformers can differ on a case or two.
- **Not shown:** every world it has seen comes from one generator. It hasn't been measured on a library written by hand or on a real organisation's notes. Your own pages must follow the format in `knowledge/distributor-wiki/`: front matter with `id`, `when`, `what`; one `§anchor` statement per line; links as `[[page-id]]`.
- Why the results are written inline rather than sent as `tool_calls`: one of the project's adapters scored 11/90 served through `tool_calls` and 90/90 served the way it was trained ([`docs/RECORD.md`](https://github.com/EvolvingAgentsLabs/lora-kernel/blob/main/docs/RECORD.md)).